https://www.ncei.noaa.gov/access/metadata/landing-page/bin/iso?id=gov.noaa.ncdc:C01589

Nclim-grid

In [1]:
import pandas as pd
import rasterio

# Define directory paths for Kaggle environment.
base_dir = r"/kaggle/input/eyds-base-dataset"  # Base directory for input datasets.
sub_dir = r"/kaggle/working/"  # Submission directory for output files.

# Define file paths for the TIFF files.
nclimgrid_path = f"{base_dir}/nclimgrid-monthly-202107.tif"

# Define file paths for training and validation CSV datasets.
train_file = f"{base_dir}/Training_data.csv"
valid_file = f"{base_dir}/Validation_data.csv"

# Define output file paths for the updated datasets.
output_train_csv = f"{sub_dir}/Training_data_with_tif.csv"
output_valid_csv = f"{sub_dir}/Validation_data_with_tif.csv"

In [2]:
# Read the CSV datasets
training_df = pd.read_csv(train_file)
validation_df = pd.read_csv(valid_file)

def extract_all_band_values(df, raster_path, lon_col="Longitude", lat_col="Latitude"):
    """
    Extracts values for all bands from a raster file for each (lon, lat) in the DataFrame.
    Returns a dictionary where each key corresponds to a band (e.g., 'nclimgrid_band1')
    and the value is a list of extracted pixel values.
    """
    # Create list of (lon, lat) coordinates
    coords = [(x, y) for x, y in zip(df[lon_col], df[lat_col])]
    print(f"\nDEBUG: Extracting all band values from {raster_path}")
    print(f"DEBUG: Number of coordinates: {len(coords)}")
    
    with rasterio.open(raster_path) as src:
        print(f"DEBUG: Raster info - Bands: {src.count}, CRS: {src.crs}")
        # Prepare a dictionary to store lists for each band
        band_values = {f"nclimgrid_band{i+1}": [] for i in range(src.count)}
        
        # Loop through each coordinate and sample pixel values from all bands
        for i, pixel_vals in enumerate(src.sample(coords)):
            if i < 5:
                print(f"DEBUG: Coord {coords[i]} -> extracted values: {pixel_vals}")
            for band_idx, band_val in enumerate(pixel_vals):
                band_values[f"nclimgrid_band{band_idx+1}"].append(band_val)
        print(f"DEBUG: Total values extracted for each band: {[len(vals) for vals in band_values.values()]}")
    
    return band_values

In [3]:
# Extract values for all bands from the nclimgrid TIFF for both datasets
nclim_values_train = extract_all_band_values(training_df, nclimgrid_path)
nclim_values_val = extract_all_band_values(validation_df, nclimgrid_path)

# Add the extracted band values as new columns to the DataFrames
for band_col, values in nclim_values_train.items():
    training_df[band_col] = values

for band_col, values in nclim_values_val.items():
    validation_df[band_col] = values


training_df.to_csv(output_train_csv, index=False)
validation_df.to_csv(output_valid_csv, index=False)


DEBUG: Extracting all band values from /kaggle/input/eyds-base-dataset/nclimgrid-monthly-202107.tif
DEBUG: Number of coordinates: 11229


/usr/local/lib/python3.10/dist-packages/rasterio/sample.py:95: RuntimeWarning: invalid value encountered in greater
  data = read(indexes, window=win, masked=masked)
/usr/local/lib/python3.10/dist-packages/rasterio/sample.py:95: RuntimeWarning: invalid value encountered in less
  data = read(indexes, window=win, masked=masked)


DEBUG: Raster info - Bands: 4, CRS: EPSG:4326
DEBUG: Coord (-73.90916667, 40.81310667) -> extracted values: [221.08008   20.450195  29.080078  24.769531]
DEBUG: Coord (-73.90918667, 40.813045) -> extracted values: [221.08008   20.450195  29.080078  24.769531]
DEBUG: Coord (-73.909215, 40.81297833) -> extracted values: [221.08008   20.450195  29.080078  24.769531]
DEBUG: Coord (-73.90924167, 40.81290833) -> extracted values: [221.08008   20.450195  29.080078  24.769531]
DEBUG: Coord (-73.90925667, 40.812845) -> extracted values: [221.08008   20.450195  29.080078  24.769531]
DEBUG: Total values extracted for each band: [11229, 11229, 11229, 11229]

DEBUG: Extracting all band values from /kaggle/input/eyds-base-dataset/nclimgrid-monthly-202107.tif
DEBUG: Number of coordinates: 1040
DEBUG: Raster info - Bands: 4, CRS: EPSG:4326
DEBUG: Coord (-73.971665, 40.78876333) -> extracted values: [234.38965   20.669922  29.179688  24.929688]
DEBUG: Coord (-73.97192833, 40.788875) -> extracted values

In [4]:
print("\nUpdated CSV files saved:")
print(output_train_csv)
print(output_valid_csv)


Updated CSV files saved:
/kaggle/working//Training_data_with_tif.csv
/kaggle/working//Validation_data_with_tif.csv
